This notebook uses the patches and the full time series to create a bunch of directories each which (willl) correspond to an item in the torch dataset (see next notebook). It's not performant to read a bunch of separate tiff files in a directory, but it's very easy to load/create a torch dataset that is organized in this way. The next notebook will take this format and serialize it further so it can be easily loaded for training.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import geopandas as gpd
import rasterio
import pandas as pd
from pathlib import Path
import numpy as np
from rasterio.windows import Window
from rasterio.profiles import DefaultGTiffProfile
import json
from tqdm.auto import tqdm
from rasterio.transform import xy
from affine import Affine
import concurrent.futures
from utils.io import generate_files_for_dataset_for_single_burst_id, generate_files_from_many_burst_dfs

/u/duvel-d2/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Parameters

In [3]:
patch_dir = Path('patches_256')

# Read all the data

In [4]:
df_ts_all = gpd.read_parquet('burst_ts_loc.parquet')
df_ts_all.head()

,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token,loc_path_copol,loc_path_crosspol
0,OPERA_L2_RTC-S1_T001-000054-IW1_20220108T18030...,T001-000054-IW1,2022-01-08 18:03:08+00:00,2022-01-08,VV+VH,1,488,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54238 9.60217, 2.30972 9.75694, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-01-08/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-01-08/OPERA_L2_...
1,OPERA_L2_RTC-S1_T001-000054-IW1_20220120T18030...,T001-000054-IW1,2022-01-20 18:03:07+00:00,2022-01-20,VV+VH,1,490,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54341 9.60227, 2.31075 9.75702, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-01-20/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-01-20/OPERA_L2_...
2,OPERA_L2_RTC-S1_T001-000054-IW1_20220201T18030...,T001-000054-IW1,2022-02-01 18:03:07+00:00,2022-02-01,VV+VH,1,492,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5437 9.60213, 2.31102 9.75687, 2.2...",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-01/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-01/OPERA_L2_...
3,OPERA_L2_RTC-S1_T001-000054-IW1_20220213T18030...,T001-000054-IW1,2022-02-13 18:03:07+00:00,2022-02-13,VV+VH,1,494,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5426 9.60194, 2.30995 9.7567, 2.27...",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-13/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-13/OPERA_L2_...
4,OPERA_L2_RTC-S1_T001-000054-IW1_20220225T18030...,T001-000054-IW1,2022-02-25 18:03:06+00:00,2022-02-25,VV+VH,1,496,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54243 9.60217, 2.30977 9.75694, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-25/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-25/OPERA_L2_...


In [5]:
burst_ids = df_ts_all.jpl_burst_id.unique().tolist()
len(burst_ids)

2671

# Make the TS Files

Want `dataset_samples_npz / <burst_id>__<burst_id>__<date_str>__<patch_index>`

and files as:

```
...npz

```

In [6]:
df_ts_list = [group_df for _, group_df in df_ts_all.groupby('jpl_burst_id')]

### Make files for one burst id

In [7]:
# %%time

# generate_files_for_dataset_for_single_burst_id(df_ts_list[0], patch_dir, max_workers=30)

### Make all

In [10]:
bad_tiffs = [336]

In [ ]:
_ = generate_files_from_many_burst_dfs(df_ts_list[337:], patch_dir, max_threads=40, n_jobs=1)

generating files:   0%|                               | 0/2334 [00:00<?, ?it/s]